In [1]:
import pandas as pd
from pathlib import Path

# =========================
# 1. File paths
# =========================

traffic_folder = Path(r"D:\KUL MSDS 1\mda_project\data\traffic")

traffic_files = [
    traffic_folder / "data-2025-10.csv",
    traffic_folder / "data-2025-11.csv",
    traffic_folder / "data-2025-12.csv",
]

output_path = traffic_folder / "traffic_q4_2025_aggregated.csv"

# Raw traffic files have no header, so we assign column names manually
traffic_columns = ["siteid", "richting", "type", "van", "tot", "aantal"]

# =========================
# 2. Process each monthly file
# =========================

all_agg = []

for file in traffic_files:
    print(f"\nProcessing {file.name}...")

    # Read raw file without header
    df = pd.read_csv(
        file,
        header=None,
        names=traffic_columns
    )

    print("Raw shape:", df.shape)

    # Convert timestamps
    df["van"] = pd.to_datetime(df["van"], errors="coerce")
    df["tot"] = pd.to_datetime(df["tot"], errors="coerce")

    # Remove invalid timestamp rows
    df = df[df["van"].notna()]

    # Convert count to numeric
    df["aantal"] = pd.to_numeric(df["aantal"], errors="coerce")
    df = df[df["aantal"].notna()]

    # Keep only cyclist rows
    df = df[df["type"].astype(str).str.upper().eq("FIETSERS")]

    # Keep only Q4 2025
    df = df[(df["van"] >= "2025-10-01") & (df["van"] < "2026-01-01")]

    # Create time features
    df["year"] = df["van"].dt.year
    df["month"] = df["van"].dt.month
    df["day"] = df["van"].dt.day
    df["hour"] = df["van"].dt.hour
    df["weekday"] = df["van"].dt.day_name()
    df["is_weekend"] = df["van"].dt.dayofweek >= 5

    # Aggregate to hourly level per site and direction
    agg = (
        df.groupby(
            [
                "siteid",
                "richting",
                "year",
                "month",
                "day",
                "hour",
                "weekday",
                "is_weekend",
            ],
            as_index=False
        )["aantal"]
        .sum()
        .rename(columns={"aantal": "total_traffic"})
    )

    print("Aggregated shape:", agg.shape)

    all_agg.append(agg)

# =========================
# 3. Combine all monthly aggregations
# =========================

traffic_agg = pd.concat(all_agg, ignore_index=True)

# Re-aggregate in case duplicate groups exist across files
traffic_agg = (
    traffic_agg.groupby(
        [
            "siteid",
            "richting",
            "year",
            "month",
            "day",
            "hour",
            "weekday",
            "is_weekend",
        ],
        as_index=False
    )["total_traffic"]
    .sum()
)

# =========================
# 4. Save final aggregated file
# =========================

traffic_agg.to_csv(output_path, index=False)

print("\nDone!")
print("Saved to:", output_path)
print("Rows:", len(traffic_agg))
print(traffic_agg.head())


Processing data-2025-10.csv...
Raw shape: (863040, 6)
Aggregated shape: (211478, 9)

Processing data-2025-11.csv...
Raw shape: (835200, 6)
Aggregated shape: (206874, 9)

Processing data-2025-12.csv...
Raw shape: (863040, 6)
Aggregated shape: (212220, 9)

Done!
Saved to: D:\KUL MSDS 1\mda_project\data\traffic\traffic_q4_2025_aggregated.csv
Rows: 630572
   siteid richting  year  month  day  hour    weekday  is_weekend  \
0       1       IN  2025     10    1     0  Wednesday       False   
1       1       IN  2025     10    1     1  Wednesday       False   
2       1       IN  2025     10    1     2  Wednesday       False   
3       1       IN  2025     10    1     3  Wednesday       False   
4       1       IN  2025     10    1     4  Wednesday       False   

   total_traffic  
0            1.0  
1            1.0  
2            0.0  
3            2.0  
4            3.0  
